# 02 用 pymatgen 表示周期材料结构

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/02_pymatgen_structure.ipynb)

## Learning objectives
理解 lattice、fractional/Cartesian coordinates、periodic boundary conditions、composition、density 和 neighbor environment，并能读取 CIF。

In [ ]:
!pip -q install pymatgen

## 1. 为什么 CIF 不能直接作为普通表格输入？
晶体/COF 是周期结构：原子位置依赖晶格，边界两侧的原子也可能互为邻居。机器学习前必须先把 CIF 解析为包含 **lattice + species + coordinates + PBC** 的结构对象。

In [ ]:
from pymatgen.core import Lattice, Structure
lattice=Lattice.hexagonal(a=12.0,c=3.5)
structure=Structure(lattice,['C','C','N','N'],[[0,0,0.5],[0.5,0.5,0.5],[0.25,0.25,0.5],[0.75,0.75,0.5]])
print(structure)
print('Formula:',structure.composition.reduced_formula)
print('Volume (A^3):',structure.volume)
print('Density:',structure.density)

In [ ]:
for i,site in enumerate(structure):
    print(i, site.species_string, 'frac=',site.frac_coords, 'cart=',site.coords)

## 2. 读取真实 CIF
科研中通常使用：
```python
from pymatgen.core import Structure
s = Structure.from_file('your_cof.cif')
```
读取后先检查 formula、atom count、lattice、volume、density，再进入描述符/GNN。**不要默认数据库 CIF 一定干净。**

In [ ]:
# 周期邻居示例
center = structure[0]
neighbors = structure.get_neighbors(center, r=6.0)
for n in neighbors[:10]:
    print(n.species_string, 'distance=',round(n.nn_distance,3))

## 3. COF 特别需要检查什么？
- 是否包含溶剂/客体；
- H 原子是否完整；
- disorder/occupancy 是否合理；
- 层间距离和 stacking 是否符合研究对象；
- primitive/conventional cell 是否一致；
- 几何优化前后结构是否混用。

这些问题会直接影响后面的 pore descriptors、graph edges 和 ML labels。

## Exercises
1. 修改 `a` 和 `c`，观察体积和密度变化。
2. 将一个 N 替换为 O，观察 composition。
3. 修改 neighbor cutoff，记录邻居数变化。
4. 思考：只看 composition 能否识别 AA stacking 与 slipped stacking？为什么？

### Take-home message
材料结构不是一行化学式。对 COF，周期性、孔道几何和层间堆积都是 representation 的一部分。